# EXAM CHEAT SHEET — Problem 1
## Pipeline: Load → PCA (5D→1D) → Plot → Polynomial Regression → Predict

**GOLDEN RULE**: `mu_X`, `sigma_X`, `U1`, `W` are computed from TRAINING data only. Apply to test data without recomputing.

**Allowed**: numpy docs, this notebook. **Not allowed**: scikit-learn.

In [1]:
import numpy as np
import matplotlib.pyplot as plt

TRAIN_PATH = 'train.csv'
TEST_PATH  = 'test.csv'

---
## 1(a) — Load data, verify shape
- `genfromtxt` reads CSV → numpy array
- `skip_header=1` skips the column-name row
- train has 6 cols (x0–x4 + t), test has 5 cols (x0–x4 only)

In [2]:
# --- 1(a) ---
train_data = np.genfromtxt(TRAIN_PATH, delimiter=',', skip_header=1)
test_data  = np.genfromtxt(TEST_PATH,  delimiter=',', skip_header=1)

X_train = train_data[:, :5]   # columns 0-4, shape (m, 5)
t_train = train_data[:, 5]    # column  5,   shape (m,)
X_test  = test_data            # shape (m_test, 5)

print('X_train:', X_train.shape)   # expect (50, 5)
print('t_train:', t_train.shape)   # expect (50,)
print('X_test: ', X_test.shape)    # expect (10, 5)

FileNotFoundError: train.csv not found.

---
## 1(b) — PCA: reduce 5D → 1D
**Algorithm: Principal Component Analysis (PCA)**

**Why normalize?** Features have different scales (x0 ~0-30, x3 ~0-200). Without normalization x3 dominates just because of its magnitude, not information content. Z-score sets all to mean=0, std=1.

**Steps**: Normalize → Covariance matrix → Eigendecomposition → Sort by eigenvalue desc → Project onto PC1

**Key shapes**:
- `X_train_norm`: (50,5) — normalized features  
- `Sigma`: (5,5) — covariance matrix  
- `eigenvectors`: (5,5) — columns are principal directions  
- `U1`: (5,1) — first principal direction  
- `x1_train`: (50,) — 1D projection of all training points

In [ ]:
# --- 1(b) ---
# STEP 1: Normalize with TRAINING stats only
mu_X    = X_train.mean(axis=0)   # (5,)  — axis=0 means across rows (per column)
sigma_X = X_train.std(axis=0)    # (5,)
X_train_norm = (X_train - mu_X) / sigma_X   # (50, 5)

print('Means (should be ~0):', X_train_norm.mean(axis=0).round(10))
print('Stds  (should be ~1):', X_train_norm.std(axis=0).round(10))

In [ ]:
# STEP 2: Covariance matrix  Σ = (1/m) * X_norm.T @ X_norm
# Off-diagonal values near 1 = features are highly correlated
m = X_train_norm.shape[0]
Sigma = (1/m) * X_train_norm.T @ X_train_norm   # (5,5)
print('Covariance matrix:\n', Sigma.round(3))

In [ ]:
# STEP 3: Eigendecomposition
# eigenvalues  = variance captured by each direction
# eigenvectors = the directions (each COLUMN is one PC)
eigenvalues, eigenvectors = np.linalg.eig(Sigma)

# STEP 4: Sort by eigenvalue descending (largest variance first)
order        = np.argsort(eigenvalues)[::-1]   # argsort=ascending, [::-1]=reverse
eigenvalues  = eigenvalues[order].real         # .real drops tiny imaginary rounding noise
eigenvectors = eigenvectors[:, order].real     # sort COLUMNS (each col = one PC direction)

for i, lam in enumerate(eigenvalues):
    pct = lam / eigenvalues.sum() * 100
    print(f'  PC{i+1}: eigenvalue={lam:.4f}  ({pct:.1f}% of variance)')

In [ ]:
# STEP 5: Project onto PC1
# U1 is the first column of sorted eigenvectors — the direction of maximum variance
U1       = eigenvectors[:, :1]              # shape (5, 1)
x1_train = (X_train_norm @ U1).flatten()   # (50,5) @ (5,1) = (50,1) → flatten → (50,)

print('x1_train shape:', x1_train.shape)
print(f'PC1 explains {eigenvalues[0]/eigenvalues.sum()*100:.1f}% of variance')

---
## 1(c) — Plot t vs x1, choose polynomial degree
- Straight line → degree 1
- One bend (U/∩) → degree 2  ← this dataset
- Two bends (S-shape) → degree 3

In [ ]:
# --- 1(c) ---
plt.figure(figsize=(7, 5))
plt.scatter(x1_train, t_train, color='steelblue', edgecolors='navy', s=40, alpha=0.8)
plt.xlabel('x1  (PC1 projection)')
plt.ylabel('t  (target)')
plt.title('t vs x1 — choose polynomial degree from this plot')
plt.grid(True, alpha=0.3)
plt.show()

# Look at the shape:
#   - One bend (quadratic curve) → DEGREE = 2
#   - Two bends (S-curve)        → DEGREE = 3
print('Decision: the relationship looks quadratic → DEGREE = 2')

---
## 1(d) — Polynomial Regression + Uncertainty

**Design matrix Φ**: each row = `[1, x, x², ..., x^d]`  
The `1` in column 0 is the bias (handles the intercept w₀).

**Normal Equation** (closed-form, no gradient descent needed):  
`W = (ΦᵀΦ)⁻¹ Φᵀ t`  
Derived by setting gradient of MSE = 0 and solving for W.

**Uncertainty = RMSE**:  
`σ = sqrt(mean((t - t_pred)²))`  
±1σ covers ~68% of points, ±2σ covers ~95%.

In [ ]:
# --- 1(d) ---
DEGREE = 2   # <-- change based on what you see in 1(c)

def build_design_matrix(x, degree):
    # Input:  x     shape (m,)
    # Output: Phi   shape (m, degree+1)
    # Row i = [x[i]^0, x[i]^1, ..., x[i]^degree] = [1, x, x², ...]
    return np.column_stack([x**i for i in range(degree + 1)])

Phi_train = build_design_matrix(x1_train, DEGREE)   # (50, 3) for degree=2
print('Phi_train shape:', Phi_train.shape)
print('First row (should be [1, x, x²]):', Phi_train[0].round(4))

In [ ]:
# Normal equation: W = inv(Phi.T @ Phi) @ Phi.T @ t
W = np.linalg.inv(Phi_train.T @ Phi_train) @ Phi_train.T @ t_train
print('Weights W:', W.round(4))
# W[0]=intercept, W[1]=linear coeff, W[2]=quadratic coeff

In [ ]:
# Predictions on training data
t_pred_train = Phi_train @ W

# Uncertainty = RMSE (Root Mean Squared Error)
residuals  = t_train - t_pred_train
sigma_pred = np.sqrt(np.mean(residuals**2))

print(f'RMSE (expected uncertainty): ±{sigma_pred:.4f}')
print(f'95% interval: ±{2*sigma_pred:.4f}')

In [ ]:
# Plot: data + fit + uncertainty band
x_line   = np.linspace(x1_train.min() - 0.1, x1_train.max() + 0.1, 300)
t_line   = build_design_matrix(x_line, DEGREE) @ W

plt.figure(figsize=(9, 5))
plt.fill_between(x_line, t_line - sigma_pred, t_line + sigma_pred,
                 alpha=0.2, color='red', label=f'±1σ = ±{sigma_pred:.2f}')
plt.fill_between(x_line, t_line - 2*sigma_pred, t_line + 2*sigma_pred,
                 alpha=0.1, color='red', label='±2σ (95%)')
plt.scatter(x1_train, t_train, color='steelblue', edgecolors='navy',
            s=40, alpha=0.8, zorder=5, label='Training data (t)')
plt.plot(x_line, t_line, 'r-', linewidth=2.5, zorder=4,
         label=f'Degree-{DEGREE} fit')
plt.xlabel('x1 (PC1)')
plt.ylabel('t')
plt.title(f'Polynomial regression degree {DEGREE} — RMSE={sigma_pred:.3f}')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

---
## 1(e) — Predict test dataset

Apply the **exact same pipeline** using training statistics:
1. Normalize with training `mu_X`, `sigma_X` (not test stats!)
2. Project with training `U1`
3. Build design matrix, multiply by training `W`

**Never recompute from test data** — test is unseen, you don't have its ground truth.

In [ ]:
# --- 1(e) ---
# Step 1: normalize with TRAINING mu_X and sigma_X
X_test_norm = (X_test - mu_X) / sigma_X

# Step 2: project onto PC1 with TRAINING eigenvectors U1
x1_test = (X_test_norm @ U1).flatten()

# Step 3: build design matrix and predict
Phi_test    = build_design_matrix(x1_test, DEGREE)
t_test_pred = Phi_test @ W

print('Predicted values for test set:')
for i, (xi, ti) in enumerate(zip(x1_test, t_test_pred)):
    print(f'  Sample {i}: x1={xi:.4f}  t_pred={ti:.4f}')

---
## QUICK REFERENCE

### numpy one-liners to remember
```python
np.genfromtxt(path, delimiter=',', skip_header=1)  # load CSV
arr.mean(axis=0)          # mean per column
arr.std(axis=0)           # std per column
arr.T @ arr               # matrix multiply (A-transpose times A)
np.linalg.eig(M)          # eigenvalues, eigenvectors
np.argsort(v)[::-1]       # indices for descending sort
eigenvectors[:, order]    # reorder columns
np.column_stack([...])    # build design matrix
np.linalg.inv(M)          # matrix inverse
np.sqrt(np.mean(r**2))    # RMSE
np.linspace(a, b, N)      # N evenly spaced points from a to b
plt.fill_between(x, y1, y2, alpha=0.2)  # uncertainty band
```

### PCA checklist (in order)
1. `mu_X = X_train.mean(axis=0)` — mean per feature
2. `sigma_X = X_train.std(axis=0)` — std per feature  
3. `X_norm = (X - mu_X) / sigma_X` — normalize
4. `Sigma = (1/m) * X_norm.T @ X_norm` — covariance (5×5)
5. `eigenvalues, eigenvectors = np.linalg.eig(Sigma)` — decompose
6. `order = np.argsort(eigenvalues)[::-1]` — sort descending
7. `U1 = eigenvectors[:, :1]` — first PC direction (5×1)
8. `x1 = (X_norm @ U1).flatten()` — project to 1D

### Polynomial regression checklist (in order)
1. `Phi = np.column_stack([x**i for i in range(degree+1)])` — design matrix
2. `W = np.linalg.inv(Phi.T @ Phi) @ Phi.T @ t` — normal equation
3. `t_pred = Phi @ W` — predict
4. `sigma = np.sqrt(np.mean((t - t_pred)**2))` — RMSE = uncertainty

### Test data pipeline (use TRAINING stats throughout)
```python
X_test_norm  = (X_test - mu_X) / sigma_X    # training mu_X, sigma_X
x1_test      = (X_test_norm @ U1).flatten() # training U1
Phi_test     = build_design_matrix(x1_test, DEGREE)
t_test_pred  = Phi_test @ W                 # training W
```

### Laplace distribution (Problem 2 — for reference)
```python
# A = 1 / (lam * sqrt(2))
# Mean = mu
# Variance = lam^2

rng = np.random.default_rng(seed=42)
u1 = rng.uniform(0, 1, N)
u2 = rng.uniform(0, 1, N)
s  = np.where(u1 < 0.5, 1, -1)             # sign
x  = s * lam * np.log(u2) / np.sqrt(2)     # sample

# PDF for plotting:
A   = 1 / (lam * np.sqrt(2))
pdf = A * np.exp(-np.sqrt(2) * np.abs(x_plot - mu) / lam)
```